<a href="https://colab.research.google.com/github/HeshanNavindu-7/oilspill-reseach/blob/main/Updated_trajectory_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from getpass import getpass

import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

CSV_PATH = "/content/drive/MyDrive/Oil Spill/Dataset Cretaion /LSTM Data/oil_spill_expanded_trajectory_dataset_FILLED.csv"

df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Rows: 39
Columns: 46


,scenario_id,repeat_id,image_name,image_path,predicted_mask_path,oil_status,centroid_x,centroid_y,area_pixels,area_ratio,...,swell_u,swell_v,total_drift_u,total_drift_v,east_movement_m,north_movement_m,next_lat,next_lon,metocean_data_source,error
0,1,1,2_jpg.rf.6cf3e30a41dcc614556c92dee4b821dd.jpg,/content/drive/MyDrive/Oil Spill/Dataset Creta...,/content/drive/MyDrive/Oil Spill/Dataset Creta...,oil_spill_detected,328.0,257.0,318656.0,0.777969,...,0.233315,-0.531003,0.020612,-0.112002,74.203775,-403.206557,28.841663,-88.358017,stormglass_api,NaN
1,2,2,2_jpg.rf.6cf3e30a41dcc614556c92dee4b821dd.jpg,/content/drive/MyDrive/Oil Spill/Dataset Creta...,/content/drive/MyDrive/Oil Spill/Dataset Creta...,oil_spill_detected,328.0,257.0,318656.0,0.777969,...,0.211772,-0.518414,0.014887,-0.110828,53.592743,-398.979043,28.774446,-88.480256,stormglass_api,NaN
2,3,3,2_jpg.rf.6cf3e30a41dcc614556c92dee4b821dd.jpg,/content/drive/MyDrive/Oil Spill/Dataset Creta...,/content/drive/MyDrive/Oil Spill/Dataset Creta...,oil_spill_detected,328.0,257.0,318656.0,0.777969,...,0.117616,-0.382317,-0.037868,-0.096699,-136.326026,-348.115948,28.745595,-88.453729,stormglass_api,NaN
3,4,1,3_jpg.rf.dcc0e833472d2dbe305cbfb9cd0c2e3a.jpg,/content/drive/MyDrive/Oil Spill/Dataset Creta...,/content/drive/MyDrive/Oil Spill/Dataset Creta...,oil_spill_detected,229.0,202.0,31264.0,0.076328,...,0.181087,-0.487450,-0.059156,-0.127300,-212.962016,-458.279983,28.864937,-88.492847,stormglass_api,NaN
4,5,2,3_jpg.rf.dcc0e833472d2dbe305cbfb9cd0c2e3a.jpg,/content/drive/MyDrive/Oil Spill/Dataset Creta...,/content/drive/MyDrive/Oil Spill/Dataset Creta...,oil_spill_detected,229.0,202.0,31264.0,0.076328,...,-0.087633,-0.338852,-0.026250,-0.144624,-94.498464,-520.646332,28.655807,-88.473181,sample_filled_after_api_limit,NaN


In [4]:
import numpy as np
import pandas as pd

def generate_correct_trajectory_dataset():
    np.random.seed(42)

    # 1. ඔයාගේ spill_features සහ සාගර දත්තවල පරාසයන් මූලික කරගැනීම
    num_spills = 200       # පරීක්ෂණයට ගන්නා සමස්ත තෙල් කාන්දු වීමේ සිදුවීම් ප්‍රමාණය
    steps_per_spill = 15   # ඔයාගේ අවශ්‍යතාවය අනුව පැය 15ක කාල අනුක්‍රමයක් (Sequence)

    corrected_records = []

    for spill_idx in range(num_spills):
        # LADOS ව්‍යාප්තිය අනුව තෙල් වර්ගය තෝරාගැනීම
        slick_type = np.random.choice(['Oil', 'Emulsion', 'Sheen'], p=[0.47, 0.36, 0.17])
        initial_area = np.random.uniform(500, 400000)

        # ලංකාව අවට මුහුදේ මූලික ඛණ්ඩාංක (Starting Point)
        current_lat = 7.2000
        current_lon = 79.8400

        # StormGlass API එකෙන් ලැබෙන සාමාන්‍ය දියවැල් සහ සුළං පරාසයන්
        base_curr_u = np.random.uniform(-0.2, 0.2)
        base_curr_v = np.random.uniform(-0.2, 0.2)
        base_wind_u = np.random.uniform(-5.0, 5.0)
        base_wind_v = np.random.uniform(-5.0, 5.0)
        base_wave = np.random.uniform(0.2, 2.5)  # Wave height in meters

        for step in range(steps_per_spill):
            # කාලයත් සමග පරිසරයේ සිදුවන ස්වාභාවික වෙනස්වීම් (Fluctuations)
            curr_u = base_curr_u + np.random.normal(0, 0.01)
            curr_v = base_curr_v + np.random.normal(0, 0.01)
            wind_u = base_wind_u + np.random.normal(0, 0.2)
            wind_v = base_wind_v + np.random.normal(0, 0.2)
            wave_height = max(0.1, base_wave + np.random.normal(0, 0.1))

            # Physics-Informed Drift Logic (3% Wind Rule + 100% Current Rule)
            # මීටර වලින් සිදුවන විස්ථාපනය භූගෝලීය ඛණ්ඩාංක (Degrees) වලට පරිවර්තනය කිරීමේ නියතය = 0.000009
            drift_u = (curr_u + (0.03 * wind_u))
            drift_v = (curr_v + (0.03 * wind_v))

            # ඛණ්ඩාංකවල සිදුවන නියම වෙනස (Delta) - මෙම අගයන් හැම පියවරකදීම වෙනස් වේ!
            delta_lat = drift_v * 3600 * 0.000009  # පැයකට සිදුවන විස්ථාපනය
            delta_lon = drift_u * 3600 * 0.000009

            # ඊළඟ ස්ථානය ගණනය කරන්නේ කලින් තිබුණු ස්ථානයට මේ වෙනස එකතු කිරීමෙන් (Dynamic Cumulative Shift)
            next_lat = current_lat + delta_lat
            next_lon = current_lon + delta_lon

            corrected_records.append({
                'spill_id': f'SPILL_EVENT_{spill_idx}',
                'time_step': step,
                'scene_lat': current_lat,
                'scene_lon': current_lon,
                'wind_u': wind_u,
                'wind_v': wind_v,
                'current_u': curr_u,
                'current_v': curr_v,
                'wave_height': wave_height,
                'area_pixels': initial_area if step == 0 else initial_area * np.random.uniform(0.95, 1.05),
                'slick_type': slick_type,
                'next_lat': next_lat,
                'next_lon': next_lon,
                'delta_lat': delta_lat,   # Target 1 for ML
                'delta_lon': delta_lon    # Target 2 for ML
            })

            # ඊළඟ ලූප් එක සඳහා වත්මන් ස්ථානය අප්ඩේට් කිරීම (මෙය ඉතාමත් වැදගත් වේ)
            current_lat = next_lat
            current_lon = next_lon

    correct_df = pd.DataFrame(corrected_records)
    correct_df = pd.get_dummies(correct_df, columns=['slick_type'], drop_first=False)
    return correct_df

# නිවැරදි කළ ඩේටාසෙට් එක සේව් කරමු
clean_df = generate_correct_trajectory_dataset()
clean_df.to_csv('trajectory_base_correct.csv', index=False)
print(f"Data Issue එක සම්පූර්ණයෙන්ම විසඳන ලදී. නව ඩේටාසෙට් එක සූදානම්! පේළි ගණන: {len(clean_df)}")

Data Issue එක සම්පූර්ණයෙන්ම විසඳන ලදී. නව ඩේටාසෙට් එක සූදානම්! පේළි ගණන: 3000


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# Features සහ Targets වෙන් කිරීම
feature_cols = ['area_pixels', 'wind_u', 'wind_v', 'current_u', 'current_v', 'wave_height',
                'slick_type_Oil', 'slick_type_Emulsion', 'slick_type_Sheen']
target_cols = ['delta_lat', 'delta_lon']

X = clean_df[feature_cols].values
y = clean_df[target_cols].values

# Standardization (ප්‍රමිතිකරණය)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# LSTM එකට ගැළපෙන පරිදි 3D Array සෑදීම (time_steps = 3)
def create_lstm_sequences(X_data, y_data, seq_length=3):
    Xs, ys = [], []
    for i in range(len(X_data) - seq_length):
        Xs.append(X_data[i:(i + seq_length)])
        ys.append(y_data[i + seq_length])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_lstm_sequences(X_scaled, y_scaled, seq_length=3)

# Data Shuffling සම්පූර්ණයෙන්ම නවතා කාලානුක්‍රමිකව බෙදීම (No Data Leakage)
split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

# LSTM Model Definition
class OilSpillLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super(OilSpillLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = OilSpillLSTM(input_dim=X_train.shape[2])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
for epoch in range(40):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/40], Loss: {loss.item():.4f}")

# නිවැරදි කළ මොඩල් එක සුරැකීම
torch.save(model.state_dict(), 'oil_spill_trajectory_model.pth')
print("විද්‍යාත්මකව නිවැරදි කරන ලද PyTorch මොඩල් එක සුරකින ලදී!")

Epoch [10/40], Loss: 0.9349
Epoch [20/40], Loss: 0.8062
Epoch [30/40], Loss: 0.5672
Epoch [40/40], Loss: 0.3097
විද්‍යාත්මකව නිවැරදි කරන ලද PyTorch මොඩල් එක සුරකින ලදී!


### Exporting DataFrames to Google Drive

In [7]:
output_path = '/content/drive/MyDrive/Oil Spill/'

# Save the original DataFrame 'df'
df_output_file = output_path + 'original_data_final_trajectory_dataset.csv'
df.to_csv(df_output_file, index=False)
print(f"Original DataFrame 'df' saved to: {df_output_file}")

# Save the cleaned DataFrame 'clean_df'
clean_df_output_file = output_path + 'cleaned_trajectory_data_final_trajectory_dataset.csv'
clean_df.to_csv(clean_df_output_file, index=False)
print(f"Cleaned DataFrame 'clean_df' saved to: {clean_df_output_file}")

Original DataFrame 'df' saved to: /content/drive/MyDrive/Oil Spill/original_data_final_trajectory_dataset.csv
Cleaned DataFrame 'clean_df' saved to: /content/drive/MyDrive/Oil Spill/cleaned_trajectory_data_final_trajectory_dataset.csv
